In [75]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor
)
from sklearn.neural_network import MLPRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

from category_encoders import TargetEncoder

import numpy as np
import pandas as pd

In [76]:
df=pd.read_csv('data//test.csv')

In [83]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1304 entries, 0 to 1303
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Price (INR in Lakhs)  1304 non-null   float64
 1   Area_sqft             1304 non-null   float64
 2   bathrooms             1304 non-null   float64
 3   balconies             1304 non-null   float64
 4   current_floor         1304 non-null   float64
 5   total_floors          1304 non-null   float64
 6   furnishing_status     1304 non-null   str    
 7   Mapped_Area           1304 non-null   str    
 8   facing                1304 non-null   str    
 9   property_age_bucket   1304 non-null   str    
 10  property_type         1304 non-null   str    
 11  Bedrooms              1304 non-null   float64
 12  Area_type             1304 non-null   str    
 13  Status                1304 non-null   str    
dtypes: float64(7), str(7)
memory usage: 142.8 KB


In [ ]:
# df.drop(columns=['floor_ratio'], inplace=True)

In [22]:
df.rename(columns={'Price (INR in Lakhs)':'Price',
                   'Area_sqft':'Area',
                   'bathrooms':'Bathrooms',
                   'balconies':'Balconies',
                    'furnishing_status':'Furnishing_Status',
                    'facing':'Facing',
                    'property_age_bucket':'Property_Age',
                    'property_type':'Property_Type',
                    'Area_type':'Area_Type',
                    'Status':'Property_Status',
                    'current_floor':'Current_Floor',
                    'total_floors':'Total_Floor',
                    'luxury_category':'Luxury_Category',
                    'location_advantage_category':'Location_Advantage_Category',
                    'floor_category':'Floor_Category'
                    },inplace=True
          )

In [27]:
df['BHK']=df['BHK'].astype('category')
df['Mapped_Area']=df['Mapped_Area'].astype('category')
df['Location']=df['Location'].astype('category')
df['Furnishing_Status']=df['Furnishing_Status'].astype('category')
df['Facing']=df['Facing'].astype('category')
df['Property_Age']=df['Property_Age'].astype('category')
df['Area_Type']=df['Area_Type'].astype('category')
df['Property_Status']=df['Property_Status'].astype('category')
df['Property_Type']=df['Property_Type'].astype('category')
df['Floor_Category']=df['Floor_Category'].astype('category')
df['Luxury_Category']=df['Luxury_Category'].astype('category')
df['Location_Advantage_Category']=df['Location_Advantage_Category'].astype('category')

In [52]:
columns_to_encode=['Location','Property_Type','BHK','Mapped_Area','Furnishing_Status','Facing','Property_Age','Area_Type','Property_Status','Floor_Category','Luxury_Category','Location_Advantage_Category']

In [54]:
train_df=df

In [55]:
X=train_df.drop(columns=['Price'])
y=train_df['Price']
y_transformed=np.log1p(y)

In [56]:
X = train_df.drop(columns=['Price'])
y = train_df['Price']

# One-Hot Encoding
X_linear = pd.get_dummies(
    X,
    columns=columns_to_encode,
    drop_first=True,
    dtype=int
)

print(X_linear.shape)
X_linear.head()

(1304, 140)


,Area,Bathrooms,Balconies,Current_Floor,Total_Floor,Bedrooms,"Location_Adalaj, Gandhinagar","Location_Ambapur, Gandhinagar","Location_Ambika Nagar, Kalol, Gandhinagar","Location_Bhaijipura, Gandhinagar",...,Property_Status_Under_Construction,Floor_Category_Lower Floor,Floor_Category_Mid Floor,Floor_Category_Top Floor,Luxury_Category_Luxury,Luxury_Category_Premium,Luxury_Category_Unknown,Location_Advantage_Category_Excellent,Location_Advantage_Category_Good,Location_Advantage_Category_Unknown
0,2916.0,3.0,1.0,4.0,8.0,3.0,0,0,0,0,...,0,0,1,0,0,0,1,0,0,1
1,1755.0,3.0,2.0,3.0,7.0,3.0,0,0,0,0,...,1,0,1,0,0,0,1,0,0,1
2,1908.0,3.0,2.0,4.0,8.0,3.0,0,0,0,0,...,0,0,1,0,0,0,1,0,0,1
3,2205.0,2.0,2.0,9.0,13.0,3.0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,1
4,1755.0,3.0,1.0,8.0,13.0,3.0,0,0,0,0,...,0,0,1,0,0,0,1,0,0,0


In [57]:
pearson_corr = X_linear.corrwith(y).abs().sort_values(ascending=False)

pearson_df = pd.DataFrame({
    'Feature': pearson_corr.index,
    'Pearson': pearson_corr.values
})

pearson_df.head(20)

,Feature,Pearson
0,Area,0.718954
1,Bedrooms,0.656728
2,Bathrooms,0.644726
3,BHK_4 BHK,0.558837
4,Total_Floor,0.465506
5,Property_Status_Under_Construction,0.385225
6,Mapped_Area_Gift City,0.368234
7,"Location_Gift City, Gandhinagar",0.363169
8,BHK_2 BHK,0.345584
9,Current_Floor,0.316721


In [58]:
spearman_corr = X_linear.corrwith(y, method='spearman').abs().sort_values(ascending=False)

spearman_df = pd.DataFrame({
    'Feature': spearman_corr.index,
    'Spearman': spearman_corr.values
})

spearman_df.head(20)

,Feature,Spearman
0,Bedrooms,0.705761
1,Bathrooms,0.688490
2,Area,0.688397
3,Total_Floor,0.545885
4,BHK_2 BHK,0.449306
5,BHK_3 BHK,0.436289
6,BHK_4 BHK,0.397788
7,Current_Floor,0.392137
8,Property_Status_Under_Construction,0.366509
9,Mapped_Area_Gift City,0.351224


In [59]:
from sklearn.feature_selection import mutual_info_regression

mi_scores = mutual_info_regression(
    X_linear,
    y,
    random_state=42
)

mi_df = pd.DataFrame({
    'Feature': X_linear.columns,
    'MI': mi_scores
}).sort_values('MI', ascending=False)

mi_df.head(20)

,Feature,MI
0,Area,0.751344
5,Bedrooms,0.560831
1,Bathrooms,0.543480
4,Total_Floor,0.314577
73,BHK_2 BHK,0.298738
74,BHK_3 BHK,0.267490
3,Current_Floor,0.197391
75,BHK_4 BHK,0.157470
130,Property_Status_Under_Construction,0.113188
83,Mapped_Area_Gift City,0.086392


In [60]:
from sklearn.feature_selection import f_regression

f_scores, p_values = f_regression(X_linear, y)

f_df = pd.DataFrame({
    'Feature': X_linear.columns,
    'F_Score': f_scores,
    'P_Value': p_values
}).sort_values('F_Score', ascending=False)

f_df.head(20)

,Feature,F_Score,P_Value
0,Area,1393.065040,6.290831e-208
5,Bedrooms,987.400548,9.111664e-162
1,Bathrooms,926.201590,4.247106e-154
75,BHK_4 BHK,591.265591,5.540797e-108
4,Total_Floor,360.189917,4.208059e-71
130,Property_Status_Under_Construction,226.883614,2.192530e-47
83,Mapped_Area_Gift City,204.240672,3.791796e-43
15,"Location_Gift City, Gandhinagar",197.812376,6.222789e-42
73,BHK_2 BHK,176.584602,7.005902e-38
3,Current_Floor,145.169016,9.008482e-32


In [61]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_linear)

lasso = LassoCV(
    cv=5,
    random_state=42,
    max_iter=10000
)

lasso.fit(X_scaled, y)

lasso_df = pd.DataFrame({
    'Feature': X_linear.columns,
    'Lasso_Coeff': abs(lasso.coef_)
}).sort_values('Lasso_Coeff', ascending=False)

lasso_df.head(20)

,Feature,Lasso_Coeff
0,Area,25.245008
83,Mapped_Area_Gift City,13.006750
5,Bedrooms,12.461082
75,BHK_4 BHK,9.582989
15,"Location_Gift City, Gandhinagar",7.294055
4,Total_Floor,5.343639
77,BHK_6 BHK,5.128156
45,"Location_Sector 22, Gandhinagar",4.084022
50,"Location_Sector 6, Gandhinagar",3.379211
1,Bathrooms,2.892516


In [62]:
rank_df = pd.DataFrame(index=X_linear.columns)

rank_df['Pearson'] = pearson_df.set_index('Feature')['Pearson'] \
                               .rank(ascending=False, method='average')

rank_df['Spearman'] = spearman_df.set_index('Feature')['Spearman'] \
                                 .rank(ascending=False, method='average')

rank_df['MI'] = mi_df.set_index('Feature')['MI'] \
                     .rank(ascending=False, method='average')

rank_df['F_Score'] = f_df.set_index('Feature')['F_Score'] \
                         .rank(ascending=False, method='average')

rank_df['Lasso'] = lasso_df.set_index('Feature')['Lasso_Coeff'] \
                           .rank(ascending=False, method='average')

In [63]:
rank_df['Average_Rank'] = rank_df.mean(axis=1)

final_linear_ranking = rank_df.sort_values('Average_Rank')

final_linear_ranking.head(60)

,Pearson,Spearman,MI,F_Score,Lasso,Average_Rank
Area,1.0,3.0,1.0,1.0,1.0,1.4
Bedrooms,2.0,1.0,2.0,2.0,3.0,2.0
Bathrooms,3.0,2.0,3.0,3.0,10.0,4.2
Total_Floor,5.0,4.0,4.0,5.0,6.0,4.8
BHK_4 BHK,4.0,7.0,8.0,4.0,4.0,5.4
Mapped_Area_Gift City,7.0,10.0,10.0,7.0,2.0,7.2
"Location_Gift City, Gandhinagar",8.0,11.0,13.0,8.0,5.0,9.0
BHK_2 BHK,9.0,5.0,5.0,9.0,80.0,21.6
Property_Status_Under_Construction,6.0,9.0,9.0,6.0,80.0,22.0
Current_Floor,10.0,8.0,7.0,10.0,80.0,23.0


In [64]:
from sklearn.preprocessing import OrdinalEncoder

X = train_df.drop(columns=['Price'])
y = train_df['Price']

cat_cols = columns_to_encode

X_nonlinear = X.copy()

encoder = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

X_nonlinear[cat_cols] = encoder.fit_transform(X_nonlinear[cat_cols])

X_nonlinear.head()

,Location,Area,Bathrooms,Balconies,Current_Floor,Total_Floor,Furnishing_Status,Mapped_Area,Facing,Property_Age,Property_Type,Bedrooms,BHK,Area_Type,Property_Status,Luxury_Category,Location_Advantage_Category,Floor_Category
0,36.0,2916.0,3.0,1.0,4.0,8.0,0.0,19.0,8.0,3.0,0.0,3.0,3.0,2.0,0.0,3.0,3.0,2.0
1,26.0,1755.0,3.0,2.0,3.0,7.0,0.0,13.0,8.0,3.0,0.0,3.0,3.0,2.0,1.0,3.0,3.0,2.0
2,33.0,1908.0,3.0,2.0,4.0,8.0,0.0,16.0,8.0,3.0,0.0,3.0,3.0,2.0,0.0,3.0,3.0,2.0
3,33.0,2205.0,2.0,2.0,9.0,13.0,0.0,16.0,8.0,3.0,0.0,3.0,3.0,1.0,0.0,3.0,3.0,0.0
4,55.0,1755.0,3.0,1.0,8.0,13.0,3.0,14.0,8.0,2.0,0.0,3.0,3.0,1.0,0.0,3.0,0.0,2.0


In [65]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd

rf = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_nonlinear, y)

rf_df = pd.DataFrame({
    'Feature': X_nonlinear.columns,
    'RF_Importance': rf.feature_importances_
}).sort_values('RF_Importance', ascending=False)

rf_df.head(20)

,Feature,RF_Importance
11,Bedrooms,0.404262
1,Area,0.218531
5,Total_Floor,0.193911
12,BHK,0.043060
4,Current_Floor,0.029885
2,Bathrooms,0.026413
7,Mapped_Area,0.019142
0,Location,0.018378
3,Balconies,0.007730
9,Property_Age,0.005856


In [66]:
from sklearn.ensemble import ExtraTreesRegressor

et = ExtraTreesRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

et.fit(X_nonlinear, y)

et_df = pd.DataFrame({
    'Feature': X_nonlinear.columns,
    'ET_Importance': et.feature_importances_
}).sort_values('ET_Importance', ascending=False)

et_df.head(20)

,Feature,ET_Importance
11,Bedrooms,0.206527
1,Area,0.182163
5,Total_Floor,0.141582
2,Bathrooms,0.139687
12,BHK,0.117015
14,Property_Status,0.042405
0,Location,0.036098
7,Mapped_Area,0.034699
4,Current_Floor,0.025902
3,Balconies,0.012470


In [67]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(
    random_state=42
)

gbr.fit(X_nonlinear, y)

gbr_df = pd.DataFrame({
    'Feature': X_nonlinear.columns,
    'GBR_Importance': gbr.feature_importances_
}).sort_values('GBR_Importance', ascending=False)

gbr_df.head(20)

,Feature,GBR_Importance
11,Bedrooms,0.379378
1,Area,0.292908
5,Total_Floor,0.225926
4,Current_Floor,0.022435
2,Bathrooms,0.020694
0,Location,0.017309
7,Mapped_Area,0.016372
12,BHK,0.009728
13,Area_Type,0.003541
3,Balconies,0.003068


In [68]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    rf,
    X_nonlinear,
    y,
    n_repeats=20,
    random_state=42,
    n_jobs=-1
)

perm_df = pd.DataFrame({
    'Feature': X_nonlinear.columns,
    'Permutation': perm.importances_mean
}).sort_values('Permutation', ascending=False)

perm_df.head(20)

,Feature,Permutation
11,Bedrooms,0.528887
1,Area,0.521451
5,Total_Floor,0.378306
2,Bathrooms,0.036106
4,Current_Floor,0.034893
7,Mapped_Area,0.026926
12,BHK,0.022435
0,Location,0.021237
3,Balconies,0.008052
13,Area_Type,0.006900


In [70]:
import shap
from sklearn.ensemble import RandomForestRegressor


rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_nonlinear, y)

explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_nonlinear)

shap_importance = np.abs(shap_values).mean(axis=0)

fi_shap = pd.DataFrame({
    'feature': X_nonlinear.columns,
    'shap_importance': shap_importance
}).sort_values(
    by='shap_importance',
    ascending=False
)

fi_shap

,feature,shap_importance
11,Bedrooms,19.217589
1,Area,15.848135
5,Total_Floor,14.107637
12,BHK,2.423197
4,Current_Floor,1.979401
7,Mapped_Area,1.775204
2,Bathrooms,1.712616
0,Location,1.530848
15,Luxury_Category,0.533895
9,Property_Age,0.510859


In [71]:
from sklearn.feature_selection import RFE

rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

# RFE
rfe = RFE(
    estimator=rf,
    n_features_to_select=1, 
    step=1
)

rfe.fit(X_nonlinear, y)


fi_rfe = pd.DataFrame({
    'feature': X_nonlinear.columns,
    'rfe_rank': rfe.ranking_
})

fi_rfe['rfe_score'] = 1 / fi_rfe['rfe_rank']

fi_rfe = fi_rfe.sort_values(
    by='rfe_rank',
    ascending=True
)

fi_rfe

,feature,rfe_rank,rfe_score
1,Area,1,1.000000
11,Bedrooms,2,0.500000
5,Total_Floor,3,0.333333
7,Mapped_Area,4,0.250000
12,BHK,5,0.200000
4,Current_Floor,6,0.166667
2,Bathrooms,7,0.142857
0,Location,8,0.125000
3,Balconies,9,0.111111
9,Property_Age,10,0.100000


In [69]:
from sklearn.feature_selection import mutual_info_regression

mi_scores = mutual_info_regression(
    X_nonlinear,
    y,
    random_state=42
)

mi_nonlinear_df = pd.DataFrame({
    'Feature': X_nonlinear.columns,
    'MI': mi_scores
}).sort_values('MI', ascending=False)

mi_nonlinear_df.head(20)

,Feature,MI
1,Area,0.761313
11,Bedrooms,0.571144
12,BHK,0.560880
2,Bathrooms,0.519225
0,Location,0.336458
7,Mapped_Area,0.327335
5,Total_Floor,0.281635
4,Current_Floor,0.189813
14,Property_Status,0.093295
17,Floor_Category,0.088336


In [72]:
nonlinear_df = pd.DataFrame({
    'feature': X_nonlinear.columns
})

nonlinear_df = nonlinear_df.merge(
    rf_df[['Feature','RF_Importance']],
    left_on='feature',
    right_on='Feature',
    how='left'
).drop(columns='Feature')

nonlinear_df = nonlinear_df.merge(
    et_df[['Feature','ET_Importance']],
    left_on='feature',
    right_on='Feature',
    how='left'
).drop(columns='Feature')

nonlinear_df = nonlinear_df.merge(
    gbr_df[['Feature','GBR_Importance']],
    left_on='feature',
    right_on='Feature',
    how='left'
).drop(columns='Feature')

nonlinear_df = nonlinear_df.merge(
    perm_df[['Feature','Permutation']],
    left_on='feature',
    right_on='Feature',
    how='left'
).drop(columns='Feature')

nonlinear_df = nonlinear_df.merge(
    fi_shap[['feature','shap_importance']],
    on='feature',
    how='left'
)

nonlinear_df = nonlinear_df.merge(
    fi_rfe[['feature','rfe_rank']],
    on='feature',
    how='left'
)

nonlinear_df.head()

,feature,RF_Importance,ET_Importance,GBR_Importance,Permutation,shap_importance,rfe_rank
0,Location,0.018378,0.036098,0.017309,0.021237,1.530848,8
1,Area,0.218531,0.182163,0.292908,0.521451,15.848135,1
2,Bathrooms,0.026413,0.139687,0.020694,0.036106,1.712616,7
3,Balconies,0.007730,0.012470,0.003068,0.008052,0.484775,9
4,Current_Floor,0.029885,0.025902,0.022435,0.034893,1.979401,6


In [73]:
rank_df = pd.DataFrame()

rank_df['feature'] = nonlinear_df['feature']

rank_df['RF_rank'] = nonlinear_df['RF_Importance'] \
                        .rank(ascending=False)

rank_df['ET_rank'] = nonlinear_df['ET_Importance'] \
                        .rank(ascending=False)

rank_df['GBR_rank'] = nonlinear_df['GBR_Importance'] \
                        .rank(ascending=False)

rank_df['Permutation_rank'] = nonlinear_df['Permutation'] \
                                .rank(ascending=False)

rank_df['SHAP_rank'] = nonlinear_df['shap_importance'] \
                            .rank(ascending=False)

rank_df['RFE_rank'] = nonlinear_df['rfe_rank']

In [74]:
rank_cols = [
    'RF_rank',
    'ET_rank',
    'GBR_rank',
    'Permutation_rank',
    'SHAP_rank',
    'RFE_rank'
]

rank_df['Average_Rank'] = rank_df[rank_cols].mean(axis=1)

final_nonlinear_ranking = rank_df.sort_values(
    'Average_Rank'
)

final_nonlinear_ranking

,feature,RF_rank,ET_rank,GBR_rank,Permutation_rank,SHAP_rank,RFE_rank,Average_Rank
11,Bedrooms,1.0,1.0,1.0,1.0,1.0,2,1.166667
1,Area,2.0,2.0,2.0,2.0,2.0,1,1.833333
5,Total_Floor,3.0,3.0,3.0,3.0,3.0,3,3.000000
2,Bathrooms,6.0,4.0,5.0,4.0,7.0,7,5.500000
12,BHK,4.0,5.0,8.0,7.0,4.0,5,5.500000
4,Current_Floor,5.0,9.0,4.0,5.0,5.0,6,5.666667
7,Mapped_Area,7.0,8.0,7.0,6.0,6.0,4,6.333333
0,Location,8.0,7.0,6.0,8.0,8.0,8,7.500000
3,Balconies,9.0,10.0,10.0,9.0,12.0,9,9.833333
13,Area_Type,12.0,12.0,9.0,10.0,11.0,12,11.000000
